In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import io

In [ ]:
plt.rcParams.update(plt.rcParamsDefault)

# 커스텀 팔레트
custom_cmap_logo = sns.blend_palette(['#10181e', '#e5e8e9', '#2ab1dc'], as_cmap=True) # 핏큘레이터 로고에서 따왔습니다
custom_cmap_rextreme = sns.blend_palette(['#c5d943', '#2a3457', '#0dabbe'], as_cmap=True) # 이건 렉스트림에서 따왔습니다

# 컬러맵 프리뷰용
data = np.random.randn(10, 10)
sns.heatmap(data, cmap=custom_cmap_logo)
plt.show()

# 컬러맵 프리뷰용
data = np.random.randn(10, 10)
sns.heatmap(data, cmap=custom_cmap_rextreme)
plt.show()

# 그래프 기본 테마 설정
sns.set_theme(style="whitegrid", font_scale=1) # 블루톤

# 그리드 색상 조절
plt.rcParams.update({
    "grid.color": ".8",          # 그리드 색상
    "grid.linestyle": "--",       # 그리드 점선 스타일
    "grid.linewidth": 0.8,        # 그리드 두께
    "axes.grid": True,            # 그리드 항상 켜기
    "axes.edgecolor": ".8",       # 축 테두리 색상
})

# 막대그래프 관련 설정
plt.rcParams.update({
    "lines.linewidth": 2,
    "lines.marker": "D",          # 전역 마커 설정
    "lines.markersize": 7        # 마커 크기
})

# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Nanumsquare_ac' # 나눔스퀘어
plt.rcParams['mathtext.fontset'] = 'dejavusans'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 15, 9
plt.rcParams['axes.titlesize'] = 16 # 제목 폰트 사이즈
plt.rcParams['axes.labelsize'] = 14 # 라벨 폰트 사이즈
plt.rcParams['font.size'] = 14 # 기본 폰트사이즈
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['mathtext.fontset'] = 'cm'

In [ ]:
# 알아서 째라...
def get_palette(n, cmap):
    return [cmap(i / (n - 1)) for i in range(n)]

# 파일 오픈 & 전처리

In [ ]:
def split_ga4_csv(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    tables = []
    current_table = []

    for line in lines:
        # 줄바꿈만 있거나 쉼표만 있는 빈 줄을 만나면 지금까지의 데이터를 저장
        if line.strip() == "" or line.strip().replace(',', '') == "":
            if current_table:
                # 메모리 상에서 해당 부분만 데이터프레임으로 변환
                df = pd.read_csv(io.StringIO("".join(current_table)), comment='#')
                tables.append(df)
                current_table = []
        else:
            current_table.append(line)

    # 마지막 테이블 처리
    if current_table:
        tables.append(pd.read_csv(io.StringIO("".join(current_table))))

    return tables

# 사용 예시
all_tables = split_ga4_csv('data/20260319_Rextreme_GA4/Generate_leads_overview.csv')

# GA4 overview 파일의 경우 보통:
# all_tables[0] -> 상단 요약 정보 (2컬럼)
# all_tables[1] -> 중간 상세 내역 (N컬럼)

In [ ]:
landing_df = pd.read_csv('data/20260319_Rextreme_GA4/Landing_page_Landing_page.csv', comment='#')

landing_df

## df.info()

In [ ]:
landing_df.info()

## df.describe()

In [ ]:
landing_df.describe()

In [ ]:
landing_df.describe(include='O')

## df.isna().sum()

In [ ]:
landing_df.isna().sum()

In [ ]:
not_set_counts = (landing_df == '(not set)').sum()

not_set_counts

- 아놔 결측값...

## df.head()

In [ ]:
landing_df.head()

## df.columns

In [ ]:
landing_df.columns

## df.shape

In [ ]:
landing_df.shape

In [ ]:
landing_df

## 결측값 처리

In [ ]:
# (not set)이랑 결측값이 컨트리에만 있어서 그것만 정리하고 가실게요.
landing_df['Landing page'] = landing_df['Landing page'].fillna('Other') # 결측값
landing_df['Landing page'] = landing_df['Landing page'].replace('(not set)', 'Other') # not set

(not set), NaN->Other

## 한국만 남고 나가주세요
- 다른 국가들은 한강애 500원 빠뜨린 수준이라...

In [ ]:
korea_df = landing_df[landing_df['Landing page'].str.contains("ko")]

In [ ]:
korea_df

## 특정 페이지 정리

In [ ]:
korea_df['Landing page'].value_counts()

### 체크인은 우리와 함께 할 수 없습니다

In [ ]:
# checkin 나가
korea_df = korea_df[~korea_df['Landing page'].str.contains("checkin")].copy()
landing_df = landing_df[~landing_df['Landing page'].str.contains("checkin")].copy()

korea_df

### 근데 관리자 페이지도 마찬가지임

In [ ]:
# 관리자 페이지 다 나가
korea_df = korea_df[~korea_df['Landing page'].str.contains("sensor-monitor")].copy()
landing_df = landing_df[~landing_df['Landing page'].str.contains("sensor-monitor")].copy()

korea_df = korea_df[~korea_df['Landing page'].str.contains("admin")].copy()
landing_df = landing_df[~landing_df['Landing page'].str.contains("admin")].copy()

korea_df = korea_df[~korea_df['Landing page'].str.contains("finish-screen")].copy()
landing_df = landing_df[~landing_df['Landing page'].str.contains("finish-screen")].copy()

korea_df = korea_df[~korea_df['Landing page'].str.contains("ranking-screen")].copy()
landing_df = landing_df[~landing_df['Landing page'].str.contains("ranking-screen")].copy()

korea_df = korea_df[~korea_df['Landing page'].str.contains("award-screen")].copy()
landing_df = landing_df[~landing_df['Landing page'].str.contains("award-screen")].copy()

korea_df

### 인바이트 통합
#### 일단 다 바꿔

In [ ]:
# invite 하위 페이지 다 합칠겁니다. 파이널-퓨-전-!
korea_clean_df = korea_df.copy()
korea_clean_df['Landing page'] = korea_clean_df['Landing page'].str.replace(r'/invite/.*', '/invite/*', regex=True)
korea_clean_df

In [ ]:
# invite 하위 페이지 다 합칠겁니다. 파이널-퓨-전-!
landing_clean_df = landing_df.copy()
landing_clean_df['Landing page'] = landing_clean_df['Landing page'].str.replace(r'/invite/.*', '/invite/*', regex=True)
landing_clean_df

In [ ]:
# 그룹바이에 어그리게이트맛 첨가
landing_clean_df = landing_clean_df.groupby('Landing page').agg({
    'Sessions': 'sum',
    'Active users': 'sum',
    'New users': 'sum',
    'Key events': 'sum',
    'Average engagement time per session': 'mean' # 평균 체류 시간은 평균치로 산출
}).reset_index()

## 페이지 오타 처리

In [ ]:
korea_clean_df['Landing page'] = korea_clean_df['Landing page'].str.replace(r'heat$', 'heats', regex=True) # 큐식정 렛츄고

korea_clean_df

1. /ko/heat: 404 NOT FOUND, /ko/heats: 히트 검색 페이지

In [ ]:
landing_clean_df['Landing page'] = landing_clean_df['Landing page'].str.replace(r'heat$', 'heats', regex=True) # 큐식정 렛츄고

#### 응 이제 묶자

In [ ]:
# 그룹바이에 어그리게이트맛 첨가
korea_clean_df = korea_clean_df.groupby('Landing page').agg({
    'Sessions': 'sum',
    'Active users': 'sum',
    'New users': 'sum',
    'Key events': 'sum',
    'Average engagement time per session': 'mean' # 평균 체류 시간은 평균치로 산출
}).reset_index()

In [ ]:
korea_clean_df

In [ ]:
# 그룹바이에 어그리게이트맛 첨가
landing_clean_df = landing_clean_df.groupby('Landing page').agg({
    'Sessions': 'sum',
    'Active users': 'sum',
    'New users': 'sum',
    'Key events': 'sum',
    'Average engagement time per session': 'mean' # 평균 체류 시간은 평균치로 산출
}).reset_index()

In [ ]:
# 내 하드에 저-장
korea_clean_df.to_csv('data/Landing_page_korea_preprocessing.csv', index=False)
landing_clean_df.to_csv('data/Landing_page_preprocessing.csv', index=False)

# 오래걸렸다...

## 신규 유저, 세션, 그리고 활성 유저

### 세션 많은 순

In [ ]:
n = korea_clean_df.shape[0]
cmap = custom_cmap_logo
df_sort = korea_clean_df.sort_values('Sessions', ascending=False)

ax = sns.barplot(df_sort, x = 'Landing page', y = 'Sessions', hue = 'Landing page', palette = get_palette(n, cmap))

for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge',
                    padding=3, fontsize=10, fontweight='bold')

plt.title('세션 많은 순')
plt.xlabel('페이지')
plt.ylabel('세션')
plt.xticks(rotation=90)

plt.show()

#### 랜딩페이지 빼고

In [ ]:
n = korea_clean_df.shape[0] -1
cmap = custom_cmap_logo
df_sort = korea_clean_df.sort_values('Sessions', ascending=False)
df_sort_cut = df_sort[1:]

ax = sns.barplot(df_sort_cut, x = 'Landing page', y = 'Sessions', hue = 'Landing page', palette = get_palette(n, cmap))

for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge',
                    padding=3, fontsize=10, fontweight='bold')

plt.title('세션 많은 순 (랜딩페이지 제외)')
plt.xlabel('페이지')
plt.ylabel('세션')
plt.xticks(rotation=90)

plt.show()

### 신규 유저 많은 순

In [ ]:
n = korea_clean_df.shape[0]
cmap = custom_cmap_logo
df_sort = korea_clean_df.sort_values('New users', ascending=False)

ax = sns.barplot(df_sort, x = 'Landing page', y = 'New users', hue = 'Landing page', palette = get_palette(n, cmap))

for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge',
                    padding=3, fontsize=10, fontweight='bold')

plt.title('신규 유저 많은 순')
plt.xlabel('페이지')
plt.ylabel('신규 유저')
plt.xticks(rotation=90)

plt.show()

#### 랜딩페이지 제외

In [ ]:
n = korea_clean_df.shape[0] -1
cmap = custom_cmap_logo
df_sort = korea_clean_df.sort_values('New users', ascending=False)
df_sort_cut = df_sort[1:]

ax = sns.barplot(df_sort_cut, x = 'Landing page', y = 'New users', hue = 'Landing page', palette = get_palette(n, cmap))

for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge',
                    padding=3, fontsize=10, fontweight='bold')

plt.title('신규 유저 많은 순 (랜딩페이지 제외)')
plt.xlabel('페이지')
plt.ylabel('신규 유저')
plt.xticks(rotation=90)

plt.show()

### 활성 유저 많은 순

In [ ]:
n = korea_clean_df.shape[0]
cmap = custom_cmap_logo
df_sort = korea_clean_df.sort_values('Active users', ascending=False)

ax = sns.barplot(df_sort, x = 'Landing page', y = 'Active users', hue = 'Landing page', palette = get_palette(n, cmap))

for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge',
                    padding=3, fontsize=10, fontweight='bold')

plt.title('활성 유저 많은 순')
plt.xlabel('페이지')
plt.ylabel('활성 유저')
plt.xticks(rotation=90)

plt.show()

#### 렌딩페이지 제외

In [ ]:
n = korea_clean_df.shape[0] -1
cmap = custom_cmap_logo
df_sort = korea_clean_df.sort_values('Active users', ascending=False)
df_sort_cut = df_sort[1:]

ax = sns.barplot(df_sort_cut, x = 'Landing page', y = 'Active users', hue = 'Landing page', palette = get_palette(n, cmap))

for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge',
                    padding=3, fontsize=10, fontweight='bold')

plt.title('활성 유저 많은 순 (랜딩페이지 제외)')
plt.xlabel('페이지')
plt.ylabel('활성 유저')
plt.xticks(rotation=90)

plt.show()

- 메인페이지 다음으로 사람들이 많이 찾는 페이지가 등록, 히트(히트 검색), 룰북이었다. 음... 등록이 어떤 단계로 이뤄지는거지?
1. registration: 참가 신청 페이지 (현재 마감됨)
2. registration-check: 등록 확인
3. final-registration: 모름. 결제 확정 페이지인가? 

## Key events
### Key events 많은 순

In [ ]:
n = korea_clean_df.shape[0]
cmap = custom_cmap_logo
df_sort = korea_clean_df.sort_values('Key events', ascending=False)

ax = sns.barplot(df_sort, x = 'Landing page', y = 'Key events', hue = 'Landing page', palette = get_palette(n, cmap))

for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge',
                    padding=3, fontsize=10, fontweight='bold')

plt.title('Key event 많은 순')
plt.xlabel('페이지')
plt.ylabel('Key event')
plt.xticks(rotation=90)

plt.show()

### 랜딩페이지 제외

In [ ]:
n = korea_clean_df.shape[0] -1
cmap = custom_cmap_logo
df_sort = korea_clean_df.sort_values('Key events', ascending=False)
df_sort_cut = df_sort[1:]

ax = sns.barplot(df_sort_cut, x = 'Landing page', y = 'Key events', hue = 'Landing page', palette = get_palette(n, cmap))

for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge',
                    padding=3, fontsize=10, fontweight='bold')

plt.title('Key event 많은 순 (랜딩페이지 제외)')
plt.xlabel('페이지')
plt.ylabel('Key event')
plt.xticks(rotation=90)

plt.show()

## 평균 체류시간

### 평균 체류시간 많은 순

In [ ]:
n = korea_clean_df.shape[0]
cmap = custom_cmap_logo
df_sort = korea_clean_df.sort_values('Average engagement time per session', ascending=False)

ax = sns.barplot(df_sort, x = 'Landing page', y = 'Average engagement time per session', hue = 'Landing page', palette = get_palette(n, cmap))

for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge',
                    padding=3, fontsize=10, fontweight='bold')

plt.title('평균 체류시간 많은 순')
plt.xlabel('페이지')
plt.ylabel('평균 체류시간')
plt.xticks(rotation=90)

plt.show()

- 의외로 평균 체류시간이 많았던 건 럭키 드로우랑 비밀번호 분실, 그리고 그 다음이 invite였습니다.

# 페이지 용도별

In [ ]:
korea_clean_df['Landing page'].value_counts()

In [ ]:
page_dic = {
    'Info':['/rulebook', '/timetable', '/faq', '/participant-guidelines', '/privacy', '/terms', '/agreement'],
    'Action':['/registration', '/final-registration', '/prize-consent'],
    'Status':['/registration-check', '/registration-lookup', '/ranking', '/heats', '/final-race-screen'],
    'Influence':['/invite/*'],
    'Bonus':['/lucky-draw'],
    'Utility':['/forgot-password', '/reset-password', '/sessions']
}

In [ ]:
reverse_dic = {path: category for category, paths in page_dic.items() for path in paths}

# 2. 분류 함수 정의 (와일드카드 '*' 처리 포함)
def classify_path(path):
    # 1단계: 딕셔너리에 정확히 일치하는지 확인
    if path in reverse_dic:
        return reverse_dic[path]

    # 2단계: 'invite/*' 처럼 패턴이 포함된 경우 처리
    if '/invite/' in path:
        return 'Influence'

    # 3단계: /ko 만 있거나 매칭 안 되는 경우
    return 'Main/Other'

# 3. 데이터프레임에 적용
# 'Landing page'가 '/ko/rulebook' 형태라면 '/ko'를 떼고 비교해야 합니다.
korea_clean_df['Classification'] = korea_clean_df['Landing page'].str.replace('/ko', '', regex=False).apply(classify_path)

# 4. 결과 확인 (기타로 빠진 게 없는지 체크)
print(korea_clean_df['Classification'].value_counts())

In [ ]:
korea_clean_df

## 용도별로 묶자

In [ ]:
classification_df = korea_clean_df.groupby('Classification').agg({
    'Sessions': 'sum',
    'Active users': 'sum',
    'New users': 'sum',
    'Key events': 'sum',
    'Average engagement time per session': 'mean' # 평균 체류 시간은 평균치로 산출
}).reset_index()

In [ ]:
classification = classification_df.query('Classification != "Main/Other"')
classification

## 세션 많은 순

In [ ]:
n = classification.shape[0]
cmap = custom_cmap_logo
df_sort = classification.sort_values('Sessions', ascending=False)

ax = sns.barplot(df_sort, x = 'Classification', y = 'Sessions', hue = 'Classification', palette = get_palette(n, cmap))

for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge',
                    padding=3, fontsize=10, fontweight='bold')

plt.title('세션 많은 순')
plt.xlabel('페이지')
plt.ylabel('세션')
plt.xticks(rotation=90)

plt.show()

- 가장 세션이 많았던 페이지 분류는 상태 조회였다. 상태 조회창 중에서는 히트 조회가 가장 많았다.

In [ ]:
# 1. 먼저 어떤 컬럼들이 있는지 확인해 보세요 (여기서 숫자 컬럼명을 찾으세요)
print("사용 가능한 컬럼들:", korea_clean_df.columns.tolist())

# 2. '세션수'가 담긴 컬럼명을 아래 '수치_컬럼'에 넣어주세요. (예: 'Sessions', 'Event count' 등)
# 여기서는 가장 확률이 높은 'Sessions' 혹은 'Event count'를 자동으로 찾도록 설정했습니다.
target_col = 'Sessions' if 'Sessions' in korea_clean_df.columns else ('Event count' if 'Event count' in korea_clean_df.columns else None)

if target_col:
    # 카테고리와 페이지별로 수치를 합산
    summary = korea_clean_df.groupby(['Classification', 'Landing page'])[target_col].sum().reset_index()

    # 각 카테고리(Classification)에서 수치가 가장 큰 행의 인덱스 찾기
    idx = summary.groupby('Classification')[target_col].idxmax()

    # 최종 결과 추출 (페이지명과 실제 수치가 같이 나옵니다)
    final_result = summary.loc[idx].reset_index(drop=True)

    print(f"\n--- [카테고리별 {target_col} 1위 페이지 및 실제 수치] ---")
    print(final_result)
else:
    print("\n숫자 데이터가 담긴 컬럼을 찾지 못했습니다. 컬럼명을 확인해 주세요!")

# 번외편: 국가별

In [ ]:
landing_clean_df

In [ ]:
landing_clean_df['extracted_nation'] = landing_clean_df['Landing page'].str.extract(r'/(ko|en|ja|jp|zh-TW|zh-CN)', expand=False)

# 국적 있는것만 필터링
nation_map = {
    'ko': '한국',
    'ja': '일본',
    'en': '영어권',
    'zh-TW': '중화권(번체)',
    'zh-CN': '중화권(간체)'
}

landing_clean_df['Nation'] = landing_clean_df['extracted_nation'].map(nation_map).fillna('기타')

landing_clean_df

In [ ]:
national_df = landing_clean_df.query('Nation != "기타"')
national_df.groupby('Nation').agg({
    'Sessions':'sum',
    'Active users':'sum',
    'New users':'sum',
    'Key events':'sum',
    'Average engagement time per session':'mean'
}).reset_index().sort_values('Sessions', ascending=False)